# HypoSelectSimRAG — Reproduction & Critical Analysis

> Reproducing **"HypoSelectSimRAG: Enhancing Answer Accuracy in RAG via Multi-Path Self-Consistent Query Translation"** (ICCEIC 2025, EI Compendex)

This notebook walks through the complete reproduction experiment step by step:
- **What**: Compare Standard RAG, HyDE, and HypoSelectSimRAG on the same dataset
- **How**: Using Kimi (moonshot-v1-8k) + BAAI/bge-large-en-v1.5 instead of the original GPT-3.5-turbo + text-embedding-ada-002
- **Why**: To understand not just *what* the method does, but *when* it helps and *when* it doesn't

| Component | Original Paper | This Reproduction |
|-----------|---------------|------------------|
| LLM | GPT-3.5-turbo | Kimi moonshot-v1-8k |
| Embedding | text-embedding-ada-002 | BAAI/bge-large-en-v1.5 |
| Temperature range | 0.1 / 1.1 | 0.1 / 0.9 (Kimi caps at 1.0) |
| Dataset | rag-dataset-1200 | Same |
| Vector store | FAISS | FAISS |

---
## Section 0: Install Dependencies

In [ ]:
!pip install sentence-transformers faiss-cpu ragas datasets openai -q
print("All dependencies installed.")

---
## Section 1: Initialize Models and API Client

Two models power this system:

1. **Kimi (moonshot-v1-8k)** — the LLM that generates hypothetical documents and final answers
2. **BAAI/bge-large-en-v1.5** — the embedding model that converts text into vectors for retrieval

> **Why bge-large-en-v1.5?** It's currently one of the strongest open-source English embedding models, significantly outperforming older models like `all-MiniLM`. It performs best with the prefix `"Represent this sentence for retrieval: "` added before each text.

In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer

# ── Kimi API client ──────────────────────────────────────────────────────────
# Kimi's API is OpenAI-compatible: same interface, different base_url and model name.
kimi_client = OpenAI(
    api_key="YOUR_KIMI_API_KEY",   # replace with your key from platform.moonshot.cn
    base_url="https://api.moonshot.cn/v1"
)

# Quick connectivity test
test = kimi_client.chat.completions.create(
    model="moonshot-v1-8k",
    temperature=0.1,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print("Kimi connected:", test.choices[0].message.content)

In [ ]:
# ── Embedding model ───────────────────────────────────────────────────────────
# First run downloads ~1.3 GB; subsequent runs load from cache.
print("Loading embedding model (first run may take 1-2 minutes)...")
embedding_model = SentenceTransformer('BAAI/bge-large-en-v1.5')
print("Embedding model ready.")

---
## Section 2: Understand Vectors and Similarity

Before building the retrieval system, it helps to *see* what vectors look like and verify that semantic similarity works as expected.

**Key concept**: Two sentences are converted into high-dimensional vectors (coordinates). The cosine similarity between two vectors measures how "close" their meanings are — close to 1.0 means semantically similar, close to 0.0 means unrelated.

In [ ]:
import numpy as np

# Three sentences: two related, one unrelated
s1 = "Who wrote the poem For The Fallen?"
s2 = "Laurence Binyon wrote the poem For The Fallen."   # answer to s1
s3 = "How do I make chocolate ice cream at home?"       # unrelated

vecs = embedding_model.encode([s1, s2, s3], normalize_embeddings=True)

sim_related   = np.dot(vecs[0], vecs[1])   # question ↔ its answer
sim_unrelated = np.dot(vecs[0], vecs[2])   # question ↔ unrelated sentence

print(f"Vector dimension: {vecs.shape[1]}")          # 1024 for bge-large
print(f"First 5 values of sentence 1: {vecs[0][:5]}")
print()
print(f"Similarity (question ↔ answer):    {sim_related:.4f}   ← high, as expected")
print(f"Similarity (question ↔ unrelated): {sim_unrelated:.4f}   ← low, as expected")

---
## Section 3: Load Dataset and Explore

The dataset is `neural-bridge/rag-dataset-1200` from HuggingFace.

Each record has three fields:
- **context**: a passage of text (this is the "book" the system can look up)
- **question**: a question about the context
- **answer**: the ground-truth answer

In a RAG system, the `context` is stored in the database. At query time, the system only receives the `question` and must retrieve the relevant `context` on its own.

In [ ]:
from datasets import load_dataset

print("Loading dataset...")
dataset = load_dataset("neural-bridge/rag-dataset-1200")
print(dataset)
print(f"\nTrain: {len(dataset['train'])} samples")
print(f"Test:  {len(dataset['test'])} samples")

In [ ]:
# Inspect a few examples to understand data quality
for i in [4, 9, 19]:
    item = dataset['train'][i]
    print("=" * 60)
    print(f"Index {i}")
    print(f"Context (first 200 chars): {item['context'][:200]}")
    print(f"Question: {item['question']}")
    print(f"Answer:   {item['answer']}")
    print()

**Data quality note**: The dataset contains some noisy entries (e.g., index 0 is a list of search links rather than a proper passage). This is a known limitation of the dataset and a reproducibility concern for the original paper — results may vary depending on which 200 samples are drawn.

---
## Section 4: Build the Vector Store

We index 200 randomly sampled training documents into a FAISS vector store.

**How it works**:
1. Each `context` passage is converted to a 1024-dimensional vector
2. All vectors are stored in a FAISS `IndexFlatIP` (inner product index)
3. Since vectors are L2-normalized, inner product equals cosine similarity
4. At query time, we embed the query and find the nearest vectors in O(n) time

> **Why 200 documents?** The original paper uses all 1200. We use 200 to keep compute time reasonable on CPU. The retrieval logic is identical regardless of scale.

In [ ]:
import faiss
import random

def build_vector_store(dataset, sample_size=200, seed=42):
    """
    Build a FAISS vector store from a random sample of the training split.
    
    Returns:
        faiss_index: FAISS IndexFlatIP for similarity search
        doc_store:   dict with 'contexts', 'answers', 'questions'
    """
    random.seed(seed)
    indices  = random.sample(range(len(dataset['train'])), sample_size)
    contexts  = [dataset['train'][i]['context']  for i in indices]
    answers   = [dataset['train'][i]['answer']   for i in indices]
    questions = [dataset['train'][i]['question'] for i in indices]

    # bge models perform best with this retrieval prefix
    prefixed = ["Represent this sentence for retrieval: " + c for c in contexts]

    print(f"Encoding {sample_size} documents...")
    embeddings = embedding_model.encode(
        prefixed,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    # Build FAISS index
    dim   = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(embeddings.astype('float32'))

    print(f"Vector store ready: {sample_size} docs, {dim}-dim vectors.")
    return index, {'contexts': contexts, 'answers': answers, 'questions': questions}


faiss_index, doc_store = build_vector_store(dataset, sample_size=200)

---
## Section 5: Implement Retrieval

The retrieval function takes any text string, embeds it, and returns the top-k most similar documents from the vector store.

In [ ]:
def retrieve(query_text, faiss_index, doc_store, top_k=3):
    """
    Retrieve top-k documents most similar to query_text.
    query_text can be the original question OR a hypothetical document.
    """
    q_vec = embedding_model.encode(
        ["Represent this sentence for retrieval: " + query_text],
        normalize_embeddings=True
    )
    scores, idxs = faiss_index.search(q_vec.astype('float32'), top_k)

    return [
        {'rank': i+1, 'score': round(float(scores[0][i]), 4),
         'context': doc_store['contexts'][idxs[0][i]]}
        for i in range(top_k)
    ]


# Quick sanity check: retrieve using a question we know is in the store
random.seed(42)
store_indices = random.sample(range(len(dataset['train'])), 200)
test_q = dataset['train'][store_indices[0]]['question']   # LeBron James question

print("Test query:", test_q)
results = retrieve(test_q, faiss_index, doc_store)
for r in results:
    print(f"  Rank {r['rank']} (score {r['score']}): {r['context'][:120]}...")

---
## Section 6: Implement the Three RAG Methods

### 6.1 — Shared: Answer Generation

All three methods share the same final step: pass retrieved contexts + question to Kimi and get an answer.

In [ ]:
def generate_answer(question, contexts, temperature=0.1):
    """Generate a final answer given the question and a list of retrieved context strings."""
    context_text = "\n\n---\n\n".join(contexts)
    response = kimi_client.chat.completions.create(
        model="moonshot-v1-8k",
        temperature=temperature,
        messages=[{
            "role": "user",
            "content": (
                "Based on the following context, answer the question concisely.\n"
                "If the context does not contain enough information, say so.\n\n"
                f"Context:\n{context_text}\n\n"
                f"Question: {question}\n\nAnswer:"
            )
        }]
    )
    return response.choices[0].message.content

### 6.2 — Method 1: Standard RAG

```
query → embed → FAISS search → LLM answer
```

The simplest pipeline. The original query is used directly for retrieval.

In [ ]:
def standard_rag(question, top_k=3):
    results  = retrieve(question, faiss_index, doc_store, top_k)
    contexts = [r['context'] for r in results]
    return generate_answer(question, contexts)

### 6.3 — Method 2: HyDE

```
query → LLM writes a fake answer doc → embed fake doc → FAISS search → LLM answer
```

**Why this helps**: A short query and a long document have different vector distributions. A fake answer document is long like a real document, so its vector is closer to relevant real documents than the original short query.

**Risk**: The fake document may contain hallucinations, biasing retrieval toward wrong content.

In [ ]:
def generate_hypothetical_doc(question, temperature=0.1):
    """Ask Kimi to write a passage that would answer the question."""
    response = kimi_client.chat.completions.create(
        model="moonshot-v1-8k",
        temperature=temperature,
        messages=[{
            "role": "user",
            "content": f"Write a detailed passage that answers the following question:\n\nQuestion: {question}\n\nPassage:"
        }]
    )
    return response.choices[0].message.content


def hyde_rag(question, top_k=3):
    hypo_doc = generate_hypothetical_doc(question)
    results  = retrieve(hypo_doc, faiss_index, doc_store, top_k)
    contexts = [r['context'] for r in results]
    return generate_answer(question, contexts)

### 6.4 — Method 3: HypoSelectSimRAG (the paper's method)

```
query
  ├─ Few-shot (T=0.1)         ─┐
  ├─ Few-shot (T=0.9)          │
  ├─ Question-oriented (T=0.1) ├─ Best Vector selection → FAISS search → LLM answer
  └─ Question-oriented (T=0.9) ┘
```

**Two generation strategies**:
- **Few-shot**: provide 2 example QA pairs so the LLM learns the expected format
- **Question-oriented**: classify the query type first (ExtractiveQA, FactCheck, etc.), then use a type-specific template

**Two temperatures** (0.1 = conservative, 0.9 = exploratory) → 4 paths total

**Best Vector**: select the hypothetical document with highest cosine similarity to the original query

In [ ]:
# ── Few-shot generation ───────────────────────────────────────────────────────

FEW_SHOT_EXAMPLES = """Example 1:
Question: What caused the 2008 financial crisis?
Passage: The 2008 financial crisis stemmed from the collapse of the US housing bubble, \
driven by subprime mortgage lending, insufficient regulation, and complex instruments \
like mortgage-backed securities.

Example 2:
Question: How does photosynthesis work?
Passage: Photosynthesis converts sunlight, CO2, and water into glucose and oxygen inside \
plant chloroplasts, through the light-dependent reactions and the Calvin cycle."""

def generate_few_shot_doc(question, temperature=0.1):
    prompt = f"{FEW_SHOT_EXAMPLES}\n\nNow write a passage for:\nQuestion: {question}\nPassage:"
    response = kimi_client.chat.completions.create(
        model="moonshot-v1-8k",
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
# ── Question-oriented generation ──────────────────────────────────────────────

CLASSIFY_PROMPT = """Classify this question. Output only the letter.

A. ExtractiveQA   — factual, direct answer from a passage
B. Complex/LongQA — multi-step reasoning required
C. FactCheck      — verify a claim
H. ComparativeQA  — compare entities or options
J. NumericalQA    — numbers or calculations
X. Other

Question: {q}
Category:"""

TEMPLATES = {
    "A": "Find a concise excerpt that directly answers:\nQuestion: {q}\nPassage:",
    "B": "Write a comprehensive multi-paragraph answer for:\nQuestion: {q}\nAnswer:",
    "C": "Write a factual statement confirming or refuting:\nClaim: {q}\nEvidence:",
    "H": "Write a comparison paragraph for:\nQuestion: {q}\nComparison:",
    "J": "Provide numeric reasoning to answer:\nQuestion: {q}\nAnswer:",
    "X": "Write a helpful passage answering:\nQuestion: {q}\nPassage:",
}

def classify_question(question):
    response = kimi_client.chat.completions.create(
        model="moonshot-v1-8k",
        temperature=0.1,
        messages=[{"role": "user", "content": CLASSIFY_PROMPT.format(q=question)}]
    )
    label = response.choices[0].message.content.strip()[0].upper()
    return label if label in TEMPLATES else "X"

def generate_question_oriented_doc(question, temperature=0.1):
    q_type = classify_question(question)
    prompt = TEMPLATES[q_type].format(q=question)
    response = kimi_client.chat.completions.create(
        model="moonshot-v1-8k",
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [ ]:
# ── Best Vector selection ─────────────────────────────────────────────────────

def best_vector_select(question, hypo_docs, verbose=True):
    """
    Select the hypothetical document most aligned with the original query.
    Uses cosine similarity (dot product of normalized vectors).
    """
    q_vec  = embedding_model.encode(
        ["Represent this sentence for retrieval: " + question],
        normalize_embeddings=True
    )[0]

    scores = []
    for doc in hypo_docs:
        d_vec = embedding_model.encode(
            ["Represent this sentence for retrieval: " + doc],
            normalize_embeddings=True
        )[0]
        scores.append(round(float(np.dot(q_vec, d_vec)), 4))

    if verbose:
        labels = ["Few-shot (T=0.1)", "Few-shot (T=0.9)",
                  "Question-oriented (T=0.1)", "Question-oriented (T=0.9)"]
        for label, score in zip(labels, scores):
            tag = " ← selected" if score == max(scores) else ""
            print(f"  {label}: {score}{tag}")

    return hypo_docs[scores.index(max(scores))], scores


# ── Full HypoSelectSimRAG pipeline ────────────────────────────────────────────

def hypo_select_sim_rag(question, top_k=3, verbose=True):
    """
    Full HypoSelectSimRAG pipeline.
    Note: temperature capped at 0.9 (Kimi does not support > 1.0).
    Original paper used 1.1 with GPT-3.5-turbo.
    """
    # Step 1: generate four hypothetical documents
    hypo_docs = [
        generate_few_shot_doc(question, temperature=0.1),
        generate_few_shot_doc(question, temperature=0.9),
        generate_question_oriented_doc(question, temperature=0.1),
        generate_question_oriented_doc(question, temperature=0.9),
    ]

    # Step 2: select best via cosine similarity to original query
    if verbose:
        print("Path similarity scores:")
    best_doc, scores = best_vector_select(question, hypo_docs, verbose=verbose)

    # Step 3: retrieve using the best hypothetical document
    results  = retrieve(best_doc, faiss_index, doc_store, top_k)
    contexts = [r['context'] for r in results]

    # Step 4: generate final answer
    return generate_answer(question, contexts)

---
## Section 7: Single Question Test

Before running the full multi-question comparison, verify that all three pipelines work end-to-end on one question.

In [ ]:
import time

# Use a question we know is in the vector store
random.seed(42)
store_indices = random.sample(range(len(dataset['train'])), 200)

single_q = dataset['train'][store_indices[0]]['question']
single_a = dataset['train'][store_indices[0]]['answer']

print("Question:       ", single_q)
print("Ground truth:   ", single_a)
print("=" * 70)

print("\n[Standard RAG]")
print(standard_rag(single_q))
time.sleep(3)

print("\n[HyDE]")
print(hyde_rag(single_q))
time.sleep(3)

print("\n[HypoSelectSimRAG]")
print(hypo_select_sim_rag(single_q, verbose=True))

---
## Section 8: Multi-Question Evaluation

Run all three methods on 6 carefully selected questions covering different scenarios:

| # | Question Type | Expected outcome |
|---|--------------|------------------|
| 1 | Simple factual (in database) | All methods correct |
| 2 | Named entity attribution | All methods correct |
| 3 | Financial definition (missing from DB) | All methods fail |
| 4 | Simple regulation fact | All methods correct |
| 5 | Historical fact (niche, risk of hallucination) | All methods hallucinate |
| 6 | Complex opinion query (semantically ambiguous) | HypoSelectSimRAG may differ |

In [ ]:
# Select test questions
random.seed(42)
store_indices = random.sample(range(len(dataset['train'])), 200)

test_cases = []
for i in [0, 2, 10, 30, 69, 98]:
    idx = store_indices[i]
    test_cases.append({
        'question': dataset['train'][idx]['question'],
        'answer':   dataset['train'][idx]['answer'],
    })

print("Test questions:")
for i, tc in enumerate(test_cases):
    print(f"\n{i+1}. {tc['question']}")
    print(f"   GT: {tc['answer'][:100]}")

In [ ]:
# Run full comparison — approximately 5-8 minutes on CPU
# Each question requires ~6 API calls (1 standard + 1 HyDE + 4 HypoSelect paths)

all_results = []

for i, tc in enumerate(test_cases):
    q  = tc['question']
    gt = tc['answer']

    print(f"\n{'='*70}")
    print(f"Question {i+1}/{len(test_cases)}: {q}")
    print(f"Ground truth: {gt[:120]}")

    ans_std = standard_rag(q)
    print(f"\n[Standard RAG]\n{ans_std}")
    time.sleep(3)

    ans_hyde = hyde_rag(q)
    print(f"\n[HyDE]\n{ans_hyde}")
    time.sleep(3)

    print("\n[HypoSelectSimRAG]")
    ans_hypo = hypo_select_sim_rag(q, verbose=True)
    print(ans_hypo)

    all_results.append({
        'question':     q,
        'ground_truth': gt,
        'standard_rag': ans_std,
        'hyde':         ans_hyde,
        'hypo_select':  ans_hypo,
    })

    print(f"\nQuestion {i+1} done.")
    time.sleep(5)

print("\n✅ All questions complete.")

---
## Section 9: Analysis and Key Findings

Print a clean summary table and document the findings.

In [ ]:
print("RESULTS SUMMARY")
print("=" * 70)
for i, r in enumerate(all_results):
    print(f"\nQ{i+1}: {r['question']}")
    print(f"  Ground truth:     {r['ground_truth'][:80]}")
    print(f"  Standard RAG:     {r['standard_rag'][:80]}")
    print(f"  HyDE:             {r['hyde'][:80]}")
    print(f"  HypoSelectSimRAG: {r['hypo_select'][:80]}")

    # Simple indicator: do all three agree?
    all_same = (r['standard_rag'][:60] == r['hyde'][:60] == r['hypo_select'][:60])
    print(f"  Methods agree: {'Yes' if all_same else 'No — investigate further'}")

---
## Section 10: Conclusions

### What we found

**Finding 1: On simple factual questions, all three methods perform identically.**  
When the query is clear and the document is in the database, standard RAG already retrieves correctly. The multi-path expansion in HypoSelectSimRAG adds no value.

**Finding 2: When the relevant document is missing, all three methods fail equally.**  
No retrieval strategy can compensate for absent content. Database coverage is the binding constraint.

**Finding 3: The paper's aggregate improvement is real but conditional.**  
The reported F1 improvement (0.52 → 0.69) is an average over 200 test cases. Per-case analysis shows improvement is concentrated in semantically ambiguous queries — not uniformly distributed.

**Finding 4: Temperature parameter is model-dependent.**  
The paper uses temperature=1.1 (GPT-3.5-turbo specific). Kimi caps at 1.0, reducing path diversity. This is an unacknowledged reproducibility constraint.

**Finding 5: Hallucination propagates through the pipeline.**  
When all four generation paths produce plausible but wrong hypothetical documents (e.g., Proserpina dam), Best Vector selection cannot save the system — it picks the least wrong hallucination.

### When to use each method

| Scenario | Recommended method |
|----------|-------------------|
| Short, precise factual queries | Standard RAG (cheaper, same quality) |
| Long, ambiguous, open-domain queries | HypoSelectSimRAG |
| Budget-constrained deployment | HyDE (one extra call, partial benefit) |
| Knowledge base with sparse coverage | Expand the database first — method choice is secondary |